Az Ollama fusson, mindkét modellel (`gemma4:e4b`, `bge-m3`). Ma sok hívás lesz, mind gondolkodás nélkül; egy kiértékelés kb. 1-2 perc. Futtasd sorban: **Shift + Enter**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, embed, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Előkészítés

A 3–5. alkalom függvényei (csak futtasd) és az 5. alkalom indexe (gyorsítótárból pár mp, különben kb. 1,5-2 perc).

In [ ]:
# A 3. alkalom függvényei: PDF → bekezdések → darabok. Csak futtasd.
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")
DARABOK_FAJL = Path("data/cache/darabok.json")


def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])
        if any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    return [{"jelentes": jelentes, "pdf_oldal": i, "oldal": nyomtatott_oldalszam(o), "bekezdesek": oldal_bekezdesei(o)}
            for i, o in enumerate(pymupdf.open(pdf_utvonal), start=1)]


def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]


ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz
        else:
            mondatok.append(resz)
    return mondatok


def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):
            reszek += [mondat] if len(mondat) <= max_hossz else mondat.split()
    darabok, aktualis = [], ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok


def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


def darabok_betoltese(fajl=DARABOK_FAJL):
    """A forrasok.csv összes jelentésének darabjai a gyorsítótárból; ha nincs meg, újra elkészíti és elmenti (kb. 10 mp)."""
    if fajl.exists():
        return json.loads(fajl.read_text(encoding="utf-8"))
    print("Nincs meg a darabok.json: újra elkészítem a 3. alkalom függvényeivel.")
    darabok = []
    for s in pd.read_csv("data/mnb/forrasok.csv", sep=";").itertuples():
        darabok += darabok_keszitese(jelentes_oldalai(kurzus.letolt(s.url, "data/mnb/" + s.fajl), s.jelentes))
    fajl.parent.mkdir(parents=True, exist_ok=True)
    fajl.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
    return darabok

In [ ]:
# A 4. alkalom függvényei: beágyazás, index, keresés, ellenőrzés a nyers PDF-ből. Csak futtasd.
INDEX_FAJL = Path("data/cache/embeddingek_rag.npz")    # saját fájl: más jelentések, mint a 4. alkalommal


def beagyaz(szovegek, adag=64):
    """Szövegek embeddingje adagokban (egy hívás = `adag` szöveg); numpy-mátrixot ad vissza."""
    vektorok = []
    for i in range(0, len(szovegek), adag):
        vektorok += embed(szovegek[i:i + adag])
    return np.array(vektorok, dtype=np.float32)


def index_betoltese(darabok, fajl=INDEX_FAJL):
    """A darabok normalizált embedding-mátrixa: a gyorsítótárból, ha ugyanezekhez a darabokhoz
    készült, különben kiszámolja és elmenti. Az i. sor az i. darabé."""
    idk = [d["id"] for d in darabok]
    if fajl.exists():
        tarolt = np.load(fajl)
        if list(tarolt["idk"]) == idk:
            print(f"Gyorsítótárból: {len(idk)} darab (a kiszámolás {float(tarolt['ido']):.0f} mp volt).")
            return tarolt["vektorok"]
    kezdes = time.perf_counter()
    vektorok = beagyaz([d["szoveg"] for d in darabok])
    vektorok = vektorok / np.linalg.norm(vektorok, axis=1, keepdims=True)   # normalizálás
    ido = time.perf_counter() - kezdes
    fajl.parent.mkdir(parents=True, exist_ok=True)
    np.savez(fajl, vektorok=vektorok, idk=np.array(idk), ido=ido)
    print(f"Kiszámolva: {len(idk)} darab, {ido:.0f} mp ({len(idk) / ido:.1f} darab/mp).")
    return vektorok


def keres(kerdes, darabok, index, k=5):
    """A kérdéshez leghasonlóbb k darab, (hasonlóság, darab) párokként, csökkenő sorrendben."""
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q                       # koszinusz-hasonlóság az összes darabbal egyszerre
    legjobbak = np.argsort(-hasonlosagok)[:k]      # a k legnagyobb hasonlóság helye
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]


def kiir(talalatok):
    """A találatok kiírása: hasonlóság, forrás, a szöveg eleje."""
    for hasonlosag, d in talalatok:
        hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
        print(f'{hasonlosag:.3f}  {d["jelentes"]}, {hol}:  {d["szoveg"][:140]}…')


forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")


def oldalak_ahol(minta, jelentesek):
    """(jelentés, PDF-oldal) párok, ahol a nyers PDF-szövegben megvan a minta (reguláris kifejezés)."""
    talalt = set()
    for s in forrasok[forrasok["jelentes"].isin(jelentesek)].itertuples():
        for i, oldal in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
            if re.search(minta, tisztit(oldal.get_text())):
                talalt.add((s.jelentes, i))
    return talalt


def van_jo_talalat(talalatok, vart_oldalak):
    """Igaz, ha a találatok között van olyan darab, amely a várt oldalak egyikéről jön."""
    return any((d["jelentes"], d["pdf_oldal"]) in vart_oldalak for _, d in talalatok)

In [ ]:
# Az 5. alkalom függvényei: a RAG-lánc és az ellenőrzései. Csak futtasd.
HONAPOK = {"01": "január", "02": "február", "03": "március", "04": "április", "05": "május",
           "06": "június", "07": "július", "08": "augusztus", "09": "szeptember", "10": "október",
           "11": "november", "12": "december"}


def forras_cimke(d):
    """Forrásmegjelölés egy darabhoz, például: MNB Inflációs jelentés, 2025. március, 62. oldal."""
    ev, honap = d["jelentes"].split("-")
    hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
    return f"MNB Inflációs jelentés, {ev}. {HONAPOK[honap]}, {hol}"


def kontextus(talalatok):
    """A modellnek szánt források: sorszám, forrásmegjelölés, aztán a darab szövege."""
    reszek = []
    for i, (hasonlosag, d) in enumerate(talalatok, start=1):
        reszek.append(f"[{i}] {forras_cimke(d)}\n{d['szoveg']}")
    return "\n\n".join(reszek)


def tokenszam(uzenetek):
    """Hány token a kérés a helyi modell szerint. Szöveget vagy üzenetlistát is elfogad.
    Egyetlen tokennyi választ kérünk, gondolkodás nélkül, hogy gyors legyen (1-2 mp)."""
    if isinstance(uzenetek, str):
        uzenetek = [{"role": "user", "content": uzenetek}]
    chat(uzenetek, gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]


RENDSZER = """Az MNB Inflációs jelentéseiből válaszolsz közgazdász hallgatóknak, magyarul, röviden.
Szabályok:
1. Csak a megadott forrásokból válaszolj. Saját tudásból ne írj számot.
2. Minden állítás után szögletes zárójelben jelöld a forrás számát, például [2].
3. Figyelj a jelentés dátumára: a régebbi jelentés előrejelzése nem tényadat. Ha a források eltérnek, a legújabb jelentésből vett tényadatot add meg.
4. Ha a források nem tartalmazzák a választ, ezt írd: "A forrásokban nincs benne." Ilyenkor ne írj számot."""


TOKEN_HATAR = 3000   # a 4096 tokenes ablakból ennyi a kérés; a többi a gondolkodásé és a válaszé


def rag_valasz(kerdes, darabok, index, k=5, rendszer=RENDSZER, gondolkodas="none"):
    """A teljes RAG-lánc: keresés → kontextus → tokenszám → modell. Szótárat ad vissza."""
    talalatok = keres(kerdes, darabok, index, k)
    while True:
        uzenetek = [{"role": "system", "content": rendszer},
                    {"role": "user", "content": f"Források:\n\n{kontextus(talalatok)}\n\nKérdés: {kerdes}"}]
        tokenek = tokenszam(uzenetek)
        if tokenek <= TOKEN_HATAR or len(talalatok) == 1:
            break
        talalatok = talalatok[:-1]           # nem fér bele: a leggyengébb találat kimarad
    print(f"A kérés: {tokenek} token, {len(talalatok)} darab (határ: {TOKEN_HATAR}, ablak: 4096)")
    kezdes = time.perf_counter()
    valasz = chat(uzenetek, gondolkodas=gondolkodas)["content"]
    return {"kerdes": kerdes, "valasz": valasz, "talalatok": talalatok, "tokenek": tokenek,
            "valasz_tokenek": utolso_hasznalat.get("completion_tokens"),
            "ido": time.perf_counter() - kezdes}


def hivatkozasok(valasz):
    """A válaszban szögletes zárójelben álló forrásszámok, növekvő sorrendben, ismétlés nélkül."""
    szamok = set()
    for belseje in re.findall(r"\[([\d,\s]+)\]", valasz):     # [2] vagy [2, 3]
        szamok.update(int(x) for x in re.findall(r"\d+", belseje))
    return sorted(szamok)


def kiir_valasz(eredmeny):
    """A válasz, alatta a hivatkozott források teljes megjelölésével (jelentés, nyomtatott oldal)."""
    print(eredmeny["valasz"])
    print("\nHivatkozott források:")
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            print(f"  [{n}] {forras_cimke(eredmeny['talalatok'][n - 1][1])}")
        else:
            print(f"  [{n}] nincs ilyen sorszámú forrás: a modell kitalálta")


_oldal_tar = {}


def oldal_szovege(jelentes, pdf_oldal):
    """Egy PDF-oldal nyers szövege, takarítva. A darabolástól független: ezzel ellenőrzünk."""
    if (jelentes, pdf_oldal) not in _oldal_tar:
        fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
        _oldal_tar[(jelentes, pdf_oldal)] = tisztit(pymupdf.open("data/mnb/" + fajl)[pdf_oldal - 1].get_text())
    return _oldal_tar[(jelentes, pdf_oldal)]


def szam_a_pdfbol(minta, jelentes):
    """A minta zárójeles részei (a keresett számok) az első olyan oldalon, ahol a minta megvan."""
    fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
    for i in range(1, len(pymupdf.open("data/mnb/" + fajl)) + 1):
        talalat = re.search(minta, oldal_szovege(jelentes, i))
        if talalat:
            return talalat.groups()
    return None


def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def alatamasztva(eredmeny, szam):
    """A szám benne van a válaszban, ÉS legalább egy hivatkozott darab nyers PDF-oldalán is."""
    if not szam_benne(szam, eredmeny["valasz"]):
        return False
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            d = eredmeny["talalatok"][n - 1][1]
            if szam_benne(szam, oldal_szovege(d["jelentes"], d["pdf_oldal"])):
                return True
    return False


def nem_tudja(valasz):
    """A válasz jelzi, hogy a forrásokban nincs benne, és nem ad meg számot (évszámot kivéve)."""
    jelzi = re.search(r"nincs benne|nem tudom|nem tartalmaz|nem szerepel", valasz.lower()) is not None
    szamok = [s for s in re.findall(r"\d+(?:[,.]\d+)?", valasz) if not re.fullmatch(r"(19|20)\d\d", s)]
    return jelzi and not szamok

In [ ]:
# Öt jelentés: a 4. alkalom négye, és a 2024. márciusi (a mintaprojekt kérdése miatt).
JELENTESEK = ["2023-12", "2024-03", "2024-09", "2025-03", "2026-09"]

osszes_darab = darabok_betoltese()      # a 3. alkalom darabjai; ha nincs meg a fájl, kb. 10 mp
darabok = [d for d in osszes_darab if d["jelentes"] in JELENTESEK]
print(f"{len(darabok)} darab a kiválasztott {len(JELENTESEK)} jelentésben")

index = index_betoltese(darabok)          # gyorsítótár nélkül kb. 1,5 perc
print("Az index alakja:", index.shape)

## 2. A tesztkészlet

Egy teszt: **kérdés**, **jelentés** (`None`, ha nincs benne), **minta** (reguláris kifejezés a PDF mondatára; a zárójeles rész a várt szám). A várt számot és oldalt a script olvassa ki a PDF-ből. Négy mintapélda: tényadat, előrejelzés egy megnevezett jelentésből, ugyanez egy másik jelentésből, nincs benne.

In [ ]:
TESZTEK = [
    # 1. Tényadat egy későbbi jelentésből (a 4. és 5. alkalom kérdése)
    {"kerdes": "Mekkora volt az infláció 2024-ben?",
     "jelentes": "2025-03", "minta": r"2024-ben az infláció (\d,\d) százalékon alakult"},
    # 2. Előrejelzés egy megnevezett jelentésből (a mintaprojekt kérdése); a minta két számot ad: a sáv két szélét
    {"kerdes": "Mekkora inflációt várt az MNB 2024-re a 2024. márciusi Inflációs jelentésben?",
     "jelentes": "2024-03", "minta": r"Az infláció 2024-ben (\d,\d)–(\d,\d) százalék"},
    # 3. Ugyanez egy másik jelentésből: más szám, ugyanarra az évre
    {"kerdes": "Mekkora inflációt várt az MNB 2024-re a 2023. decemberi Inflációs jelentésben?",
     "jelentes": "2023-12", "minta": r"infláció 2024-ben (\d,\d) – (\d,\d) százalék"},
    # 4. Nincs benne: a helyes válasz „A forrásokban nincs benne.”
    {"kerdes": "Mennyi volt az átlagos albérletár Debrecenben 2024-ben?",
     "jelentes": None, "minta": None},
    # 5–8. További kérdések ugyanígy
    {"kerdes": "Mekkora az MNB inflációs célja?",
     "jelentes": "2026-09", "minta": r"január 1-jétől (\d,\d) százalékban határozza meg"},
    {"kerdes": "Mekkora volt az infláció 2023-ban?",
     "jelentes": "2024-03", "minta": r"2023-ban az infláció (\d+,\d) százalékon alakult"},
    {"kerdes": "Mennyivel bővült a magyar gazdaság 2024-ben?",
     "jelentes": "2025-03", "minta": r"2024-ben a hazai GDP (\d,\d) százalékkal emelkedett"},
    {"kerdes": "Mekkora GDP-növekedést várt az MNB 2024-re a 2024. márciusi Inflációs jelentésben?",
     "jelentes": "2024-03", "minta": r"bővül 2024-ben, az év egészében (\d,\d)–(\d,\d) százalék"},
]

A várt értékek a nyers PDF-ből:

In [ ]:
def vart_ertekek(teszt):
    """A teszt várt oldalai és számai, a nyers PDF-ből. Ha nincs minta (nincs benne), üres mindkettő."""
    if teszt["minta"] is None:
        return {"oldalak": set(), "szamok": ()}
    return {"oldalak": oldalak_ahol(teszt["minta"], [teszt["jelentes"]]),
            "szamok": szam_a_pdfbol(teszt["minta"], teszt["jelentes"])}


nyomtatott = {(d["jelentes"], d["pdf_oldal"]): d["oldal"] for d in darabok}   # PDF-oldal → nyomtatott oldal

vart = {}
sorok = []
for t in TESZTEK:
    vart[t["kerdes"]] = v = vart_ertekek(t)
    oldalak = ", ".join(f'{j} {nyomtatott.get((j, p)) or "PDF " + str(p)}. o.' for j, p in sorted(v["oldalak"]))
    sorok.append({"kérdés": t["kerdes"], "várt szám": "–".join(v["szamok"] or ()) or "nincs benne",
                  "várt oldal": oldalak or "–"})
pd.DataFrame(sorok).set_index("kérdés")

In [ ]:
# Minden mintának meg kell lennie a PDF-ben: ha nincs, a teszt rossz, nem a modell.
ellenoriz_feltetel("vart", lambda v: all(v[t["kerdes"]]["szamok"] for t in TESZTEK if t["minta"]),
                   "minden mintát megtalált a script a megadott jelentésben, és kiolvasta a várt számot")

### Feladat 2.1 – saját tesztkérdések

Írj **két** saját tesztkérdést a `sajat_tesztek` listába. A segédfüggvény kiírja a mondatokat, amelyekben egy kifejezés szerepel; írd át a kifejezést (bér, GDP, forint, kamat, maginfláció…). A mondatban a számot cseréld `(\d,\d)`-re. A Copilot segíthet:

> A `mondatok_ahol` kimenetéből válassz egy mondatot, és írj hozzá tesztkérdést a `TESZTEK` mintájára (kérdés, jelentés, minta). A mintában a számot cseréld `(\d,\d)`-re, a mondat többi része maradjon szó szerint. Az eredményt tedd a `sajat_tesztek` listába.

Ha a Copilot nem látja a kimenetet, másold be a promptba a választott mondatot. Olvasd el: pontosan ugyanaz a mondat a mintában, mint a PDF-ben (gondolatjel, szóköz)?

Ha a szám egész része kétjegyű (például 17,6), a minta `(\d+,\d)` legyen: a `(\d,\d)` csak a 7,6-ot fogná meg. Az ellenőrző cella kiírja, mit olvasott ki a script: azt kaptad, amit vártál?

In [ ]:
def mondatok_ahol(minta, jelentesek=JELENTESEK, max_db=8):
    """Kiírja a nyers PDF-ek azon mondatait, amelyekben a minta (reguláris kifejezés) megvan."""
    db = 0
    for jelentes in jelentesek:
        fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
        for p in range(1, len(pymupdf.open("data/mnb/" + fajl)) + 1):
            for mondat in mondatokra(oldal_szovege(jelentes, p)):       # a 3. alkalom mondathatár-szabálya
                if re.search(minta, mondat):
                    print(f'{jelentes}, {nyomtatott.get((jelentes, p)) or "PDF " + str(p)}. oldal:  {mondat[:220]}')
                    db += 1
                    if db >= max_db:
                        return


mondatok_ahol(r"munkanélküliségi ráta")

In [ ]:
sajat_tesztek = [
    # {"kerdes": "...", "jelentes": "...", "minta": r"..."},
]

In [ ]:
def teszt_rendben(t):
    """Jó-e egy tesztkérdés formailag: van kérdés; ha van minta, megvan a jelentésben, és ad számot."""
    if not isinstance(t, dict) or not t.get("kerdes"):
        return False
    if t.get("minta") is None:
        return t.get("jelentes") is None
    if t.get("jelentes") not in JELENTESEK:
        return False
    szamok = szam_a_pdfbol(t["minta"], t["jelentes"])
    return bool(szamok) and all(s and re.fullmatch(r"\d+(,\d+)?", s) for s in szamok)

In [ ]:
sajat_tesztjeim = sajat_tesztek or None        # üres lista: még nincs kész
for _t in sajat_tesztek:
    if isinstance(_t, dict) and _t.get("minta") and _t.get("jelentes") in JELENTESEK:
        print(f'{_t["kerdes"]}  →  a script ezt olvassa ki: {szam_a_pdfbol(_t["minta"], _t["jelentes"])}')
ellenoriz_feltetel("sajat_tesztjeim", lambda ts: len(ts) >= 2 and all(teszt_rendben(t) for t in ts)
                   and any(t["minta"] for t in ts),
                   "legalább két kérdés; a minták megvannak a PDF-ben, és számot adnak")

## 3. A keresés mérése: jó oldal az első k-ban

Modellhívás nélkül: minden válaszolható kérdéshez a 10 legjobb találat.

In [ ]:
valaszolhato = [t for t in TESZTEK if t["minta"]]
tesztek_talalatai = {t["kerdes"]: keres(t["kerdes"], darabok, index, k=10) for t in valaszolhato}

for t in valaszolhato:
    tal = tesztek_talalatai[t["kerdes"]]
    helyek = [i for i, (_, d) in enumerate(tal, start=1) if (d["jelentes"], d["pdf_oldal"]) in vart[t["kerdes"]]["oldalak"]]
    print(f'{(str(helyek[0]) + ". hely") if helyek else "nincs a 10-ben":>14}  {t["kerdes"]}')

### Feladat 3.1 – Copilottal

> A `tesztek_talalatai` szótár minden válaszolható tesztkérdéshez (kulcs: a kérdés) a 10 legjobb találatot tartalmazza, a `vart` szótár pedig a várt oldalakat (`vart[kerdes]["oldalak"]`). Számold ki k = 1, 3, 5 és 10 esetén, a kérdések hányadánál van legalább egy találat a várt oldalakról az első k között. Használd a `van_jo_talalat` függvényt. Az eredmény a `talalati_arany` szótárba kerüljön (kulcs: k, érték: 0 és 1 közötti arány), és írd ki.

Olvasd el: az első k találatot nézi (`[:k]`), és a válaszolható kérdések számával oszt?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_vart_arany = {k: sum(van_jo_talalat(tal[:k], vart[q]["oldalak"]) for q, tal in tesztek_talalatai.items())
                  / len(tesztek_talalatai) for k in [1, 3, 5, 10]}
ellenoriz_feltetel("talalati_arany", lambda a: set(a) == set(_vart_arany)
                   and all(abs(a[k] - _vart_arany[k]) < 1e-9 for k in a),
                   "a találati arány k = 1, 3, 5, 10 esetén")

Miért nem emeljük egyszerűen 10-re a k-t? (4. alkalom: a legrosszabb esetben 6 darab fér a 3000 tokenbe.)

## 4. A válaszok mérése: három szempont

| szempont (RAGAS) | nálunk, scripttel |
|---|---|
| kontextus-relevancia | a várt oldal az első 5 találatban |
| hűség a forráshoz (*faithfulness*) | a válasz **minden száma** megvan a modellnek adott szövegben |
| válasz-relevancia → helyesség | a várt szám a válaszban; „nincs benne” kérdésnél a `nem_tudja` |

Nem LLM-bíró: ahol objektív helyes válasz van, script dönt.

### Feladat 4.1 – Copilottal: a számellenőrző

> Írj egy `nem_talalt_szamok(valasz, forras_szoveg)` függvényt, amely visszaadja azokat a számokat (szövegként, a válaszbeli sorrendben, ismétlés nélkül), amelyek szerepelnek a válaszban, de a forrás szövegében nem. A szögletes zárójeles forrásszámokat (például `[2]` vagy `[1, 3]`) hagyd ki. Számnak számít az egész és a tizedes tört is; a tizedespontot vedd tizedesvesszőnek (3.7 = 3,7). A forrásban a számot önálló számként keresd: a 13,7-ben nincs benne a 3,7.

Olvasd el: a forrásban szövegrészt keres a kód (`in`), vagy számokat hasonlít számokhoz? Mi lesz a 13,7-tel? És a `[2]`-vel?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz_fuggveny("nem_talalt_szamok", [
    (("Az infláció 2024-ben 3,7% volt [2].", "2024-ben az infláció 3,7 százalékon alakult"), []),
    (("Az MNB 3,5–5,0 százalékot várt [1].", "Az infláció 2024-ben 3,5–5,0 százalék között"), []),
    (("Az infláció 3,9% volt [1].", "Az infláció 3,7 százalékon alakult"), ["3,9"]),
    (("Az infláció 3.7 százalék volt.", "az infláció 3,7 százalékon alakult"), []),
    (("Az infláció 3,7% volt.", "Az infláció 13,7 százalék volt."), ["3,7"]),
    (("A forrásokban nincs benne.", "12 darab"), []),
])

Az én változatom (ha a 4.1 még nincs kész, ezzel megy tovább a notebook):

In [ ]:
def nem_talalt_szamok(valasz, forras_szoveg):
    """A válasz azon számai (a válaszbeli sorrendben, ismétlés nélkül), amelyek a forrásban nem szerepelnek."""
    def szamok(szoveg):
        szoveg = re.sub(r"\[[\d,\s]+\]", " ", szoveg)        # [2], [1, 3]: hivatkozás, nem adat
        szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)      # 3.7 → 3,7
        return re.findall(r"\d+(?:,\d+)?", szoveg)            # egész vagy tizedes tört

    forrasban = set(szamok(forras_szoveg))                   # halmaz: a 13,7 nem egyenlő a 3,7-tel
    hianyzo = []
    for s in szamok(valasz):
        if s not in forrasban and s not in hianyzo:
            hianyzo.append(s)
    return hianyzo

A teljes kiértékelés: 8 kérdés a RAG-láncon, a három szempontot script dönti el. **Kb. 1-2 perc.**

In [ ]:
def kiertekel(tesztek, darabok, index, k=5, rendszer=RENDSZER):
    """Lefuttatja a RAG-láncot minden tesztkérdésre, és három szempontból értékel, scripttel.
    Visszaadja a táblázatot és a teljes válaszokat."""
    sorok, valaszok = [], []
    kezdes = time.perf_counter()
    for t in tesztek:
        v = vart_ertekek(t)
        e = rag_valasz(t["kerdes"], darabok, index, k=k, rendszer=rendszer)
        modellnek_adott = kontextus(e["talalatok"]) + "\n" + t["kerdes"]
        if t["minta"] is None:
            jo_oldal, helyes = None, nem_tudja(e["valasz"])
        else:
            jo_oldal = van_jo_talalat(e["talalatok"], v["oldalak"])
            helyes = all(szam_benne(s, e["valasz"]) for s in v["szamok"])
        sorok.append({"kérdés": t["kerdes"],
                      "jó oldal (kontextus)": jo_oldal,
                      "hű a forráshoz": not nem_talalt_szamok(e["valasz"], modellnek_adott),
                      "helyes": helyes,
                      "token": e["tokenek"]})
        valaszok.append(e["valasz"])
    ido = time.perf_counter() - kezdes
    print(f"\n{len(tesztek)} kérdés, {2 * len(tesztek)} modellhívás (a fele egytokenes tokenszámlálás), {ido:.0f} mp")
    return pd.DataFrame(sorok).set_index("kérdés"), valaszok


def osszesit(tabla):
    """Hány kérdésnél teljesül az egyes szempont (a „nincs benne” kérdésnél a jó oldal nem számít)."""
    return {oszlop: f"{int(tabla[oszlop].dropna().sum())}/{tabla[oszlop].notna().sum()}"
            for oszlop in ["jó oldal (kontextus)", "hű a forráshoz", "helyes"]}

In [ ]:
tabla_elotte, valaszok_elotte = kiertekel(TESZTEK, darabok, index)
print(osszesit(tabla_elotte))
tabla_elotte

A válaszok:

In [ ]:
for kerdes, valasz in zip(tabla_elotte.index, valaszok_elotte):
    print(f"KÉRDÉS: {kerdes}\n{valasz}\n")

Melyik kérdésnél nincs jó oldal, a válasz mégis helyes? Melyik válasz hű a forráshoz, mégis rossz?

## 5. A hibalánc: darabolás → találat → válasz

Az utolsó kérdés: „Mekkora GDP-növekedést várt az MNB 2024-re a 2024. márciusi Inflációs jelentésben?”

In [ ]:
KERDES_GDP = TESZTEK[-1]["kerdes"]
print("Várt:", "–".join(vart[KERDES_GDP]["szamok"]), "százalék;", sorted(vart[KERDES_GDP]["oldalak"]))
print("A modell válasza:", valaszok_elotte[-1], "\n")

osszes = keres(KERDES_GDP, darabok, index, k=len(darabok))         # az összes darab, sorrendben
helyek = [i for i, (_, d) in enumerate(osszes, start=1) if (d["jelentes"], d["pdf_oldal"]) in vart[KERDES_GDP]["oldalak"]]
print("A jó oldal darabjai a keresésben:", helyek, "helyen (az első 5 kerül a modell elé)\n")
kiir(osszes[:5])

Olvasd el az öt találatot: van köztük táblázat? Benne van a jó szám? Mi áll a nyers oldalon, és mi maradt belőle?

In [ ]:
tabla = next(d for _, d in osszes[:5] if "összevetve más prognózisokkal" in d["szoveg"])
fajl = forrasok.loc[forrasok["jelentes"] == tabla["jelentes"], "fajl"].iloc[0]
oldal = pymupdf.open("data/mnb/" + fajl)[tabla["pdf_oldal"] - 1]
megmaradt = " ".join(oldal_bekezdesei(oldal))              # a 3. alkalom takarítása után

print(f"{forras_cimke(tabla)}, a nyers oldal blokkjai (>> = a takarítás kidobta):\n")
for blokk in oldal.get_text("blocks")[:26]:
    szoveg = tisztit(blokk[4])
    print(">> " if szoveg and szoveg not in megmaradt else "   ", szoveg[:95])

Mit dobott ki a takarítás? Mi maradt a táblázatdarabban, és mit tud belőle kiolvasni a modell? Hányadik helyen van a jó mondat?

### Feladat 5.1 – Copilottal

> Számold meg, a `darabok` listában hány darab első 60 karaktere egyezik meg szó szerint egy **másik jelentés** valamelyik darabjának első 60 karakterével. Az eredmény az `ismetlodo_db` változóba kerüljön, és írj ki néhány példát is ezekből a kezdetekből.

Olvasd el: a saját jelentésén belüli ismétlést nem számolja (például ugyanaz a felirat két oldalon)?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_kezdetek = {}
for _d in darabok:
    _kezdetek.setdefault(_d["szoveg"][:60], set()).add(_d["jelentes"])
ellenoriz("ismetlodo_db", vart=sum(len(_kezdetek[_d["szoveg"][:60]]) > 1 for _d in darabok))

Hány darab? Milyen kezdetek ismétlődnek?

## 6. Javítási kísérlet: a darab mondja meg, honnan jön

Minden darab elejére a jelentés neve, és újra beágyazzuk. (Más lehetőség: nagyobb k, szűrés a jelentésre, rendszerprompt. Miért nem ezek?) **Kb. 2 perc.**

In [ ]:
def fejleces(d):
    """A darab másolata: a szöveg elején a jelentés neve, hogy a beágyazás is „tudja”, honnan jön."""
    ev, honap = d["jelentes"].split("-")
    return dict(d, id=d["id"] + "_f", szoveg=f"MNB Inflációs jelentés, {ev}. {HONAPOK[honap]}. {d['szoveg']}")


darabok_f = [fejleces(d) for d in darabok]
print(darabok_f[0]["szoveg"][:160], "…")

index_f = index_betoltese(darabok_f, Path("data/cache/embeddingek_rag_fejlec.npz"))   # kb. 2 perc, ha nincs meg

### Feladat 6.1 – olvasás

A 7 válaszolható kérdésből hánynál lesz jó oldal az első 5-ben a javítás után? Írj egy egész számot.

In [ ]:
tipp_6_1 = None  # a None helyére írd a tipped

In [ ]:
talalatai_f = {t["kerdes"]: keres(t["kerdes"], darabok_f, index_f, k=10) for t in valaszolhato}
osszevetes = pd.DataFrame([{"k": k,
                           "előtte": sum(van_jo_talalat(tesztek_talalatai[q][:k], vart[q]["oldalak"]) for q in tesztek_talalatai),
                           "fejléccel": sum(van_jo_talalat(talalatai_f[q][:k], vart[q]["oldalak"]) for q in talalatai_f)}
                          for k in [1, 3, 5, 10]]).set_index("k")
print(f"Jó oldal az első k találatban (a {len(valaszolhato)} válaszolható kérdésből):")
print(osszevetes)
ellenoriz("tipp_6_1", vart=int(osszevetes.loc[5, "fejléccel"]))

A jó oldal helye kérdésenként (üres: nincs az első 10-ben):

In [ ]:
def elso_jo_hely(talalatok, oldalak):
    """Hányadik találat jön először a várt oldalak egyikéről (None, ha egyik sem)."""
    return next((i for i, (_, d) in enumerate(talalatok, start=1) if (d["jelentes"], d["pdf_oldal"]) in oldalak), None)


pd.DataFrame({"előtte": {q: elso_jo_hely(tesztek_talalatai[q], vart[q]["oldalak"]) for q in tesztek_talalatai},
              "fejléccel": {q: elso_jo_hely(talalatai_f[q], vart[q]["oldalak"]) for q in talalatai_f}}).astype("Int64")

In [ ]:
# A jelentést megnevező kérdéseknél (ezekért készült a javítás): a jó oldal ugyanott vagy előrébb van-e?
_megnevezi = [q for q in talalatai_f if "jelentésben" in q]
_hely = lambda tal, q: elso_jo_hely(tal, vart[q]["oldalak"]) or 99      # 99: nincs az első 10-ben
for q in _megnevezi:
    print(f"{_hely(tesztek_talalatai[q], q):>3} → {_hely(talalatai_f[q], q):>3}  {q}")
ellenoriz_feltetel("talalatai_f", lambda t: all(_hely(t[q], q) <= _hely(tesztek_talalatai[q], q) for q in _megnevezi),
                   "a jelentést megnevező kérdéseknél a fejléces indexben a jó oldal ugyanott vagy előrébb van")

És a válaszok? Ugyanaz a kiértékelés a fejléces indexszel (**kb. 1-2 perc**):

In [ ]:
tabla_utana, valaszok_utana = kiertekel(TESZTEK, darabok_f, index_f)
pd.DataFrame({"előtte": osszesit(tabla_elotte), "fejléccel": osszesit(tabla_utana)})

In [ ]:
tabla_utana

In [ ]:
for kerdes, valasz in zip(tabla_utana.index, valaszok_utana):
    print(f"KÉRDÉS: {kerdes}\n{valasz}\n")

Javított vagy rontott? Kérdésenként is olvasd el: hol javult, hol romlott, és miért? Megtartanád ezt a javítást? Melyik oszlop determinisztikus, melyik nem?

**Ha van időd** (új gyorsítótár-fájlnévvel): (1) szűrj a jelentésre, ha a kérdés megnevez egyet, és mérd meg; (2) a takarításban tartsd meg a táblázatok rövid sorait, darabold újra, és mérd meg; (3) írd át a `RENDSZER` 3. szabályát (múlt idejű kérdésre tényadat, ne előrejelzés), add át a `kiertekel(..., rendszer=...)` hívásnak, és mérd meg.

## 7. A projektben: 10 tesztkérdés

10 kérdés (párban: közösen), futtatás **előtt**: kb. 4 dokumentumos, 3 adatos, 3 kombinált, köztük legalább egy „nincs benne”. Ma a dokumentumosakat és a „nincs benne” kérdést írd meg teljesen; az adatosaknál most elég a kérdés és az, hogy melyik táblából számolja majd a script a várt értéket. Helyük a projekt repójában a `tesztkerdesek.md`, így:

```text
## 3. Mit várt a 2024. márciusi jelentés 2024-re, és mennyi lett?
- típus: kombinált
- forrás: MNB Inflációs jelentés, 2024. március, 12. oldal; KSH fogyasztóiár-index, 2024, éves átlag
- minta: r"Az infláció 2024-ben (\d,\d)–(\d,\d) százalék"
- várt: a script számolja (sáv: 3,5–5,0; tény: a KSH-táblából)
- eredmény: (a kiértékelés után)
```

### Otthonra

- Egészítsd ki a `sajat_tesztek` listát **10 kérdésre** (köztük legalább egy „nincs benne”), és ellenőrizd a 2.1 ellenőrző cellájával.
- Futtasd le rajtuk a kiértékelést mindkét indexszel: `kiertekel(sajat_tesztek, darabok, index)` és `kiertekel(sajat_tesztek, darabok_f, index_f)` (10 kérdés: indexenként kb. 1-2 perc). Hol hibázik a lánc? A hibalánc melyik pontján?

In [ ]:
# kiertekel(sajat_tesztek, darabok, index)
# kiertekel(sajat_tesztek, darabok_f, index_f)
